In [2]:
!pip install -q \
transformers \
datasets \
accelerate \
evaluate \
jiwer \
huggingface_hub \
soundfile \
librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 95.1 MB/s eta 0:00:00


In [3]:
import os
import pandas as pd
import numpy as np
import torch

from huggingface_hub import login, HfApi, list_repo_files

from datasets import Dataset, Audio

print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Torch version: 2.11.0+cpu
GPU available: False


In [4]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN is None:
    print("HF_TOKEN not found in Colab Secrets")
else:
    login(token=HF_TOKEN)
    print("Successfully logged in!")

Successfully logged in!


In [5]:
repo_id = "ARTPARK-IISc/Vaani"
api = HfApi()

info = api.dataset_info(
    repo_id=repo_id,
    token=HF_TOKEN
)
print("Dataset:", info.id)
print("Author:", info.author)
print("Gated:", info.gated)

Dataset: ARTPARK-IISc/Vaani
Author: ARTPARK-IISc
Gated: auto


In [7]:
files = list_repo_files(
    repo_id=repo_id,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Total files:", len(files))

Total files: 19329


In [8]:
hindi_files = [
    file for file in files
    if "hindi" in file.lower()
]
print("Hindi-related files found:", len(hindi_files))
for i, file in enumerate(hindi_files[:30]):
    print(i, file)

Hindi-related files found: 5682
0 audio/Hindi/train-00000-of-05620.parquet
1 audio/Hindi/train-00001-of-05620.parquet
2 audio/Hindi/train-00002-of-05620.parquet
3 audio/Hindi/train-00003-of-05620.parquet
4 audio/Hindi/train-00004-of-05620.parquet
5 audio/Hindi/train-00005-of-05620.parquet
6 audio/Hindi/train-00006-of-05620.parquet
7 audio/Hindi/train-00007-of-05620.parquet
8 audio/Hindi/train-00008-of-05620.parquet
9 audio/Hindi/train-00009-of-05620.parquet
10 audio/Hindi/train-00010-of-05620.parquet
11 audio/Hindi/train-00011-of-05620.parquet
12 audio/Hindi/train-00012-of-05620.parquet
13 audio/Hindi/train-00013-of-05620.parquet
14 audio/Hindi/train-00014-of-05620.parquet
15 audio/Hindi/train-00015-of-05620.parquet
16 audio/Hindi/train-00016-of-05620.parquet
17 audio/Hindi/train-00017-of-05620.parquet
18 audio/Hindi/train-00018-of-05620.parquet
19 audio/Hindi/train-00019-of-05620.parquet
20 audio/Hindi/train-00020-of-05620.parquet
21 audio/Hindi/train-00021-of-05620.parquet
22 audio/H

In [9]:
hindi_parquet_files = [
    file for file in hindi_files
    if file.endswith(".parquet")
]

print("Hindi Parquet files:", len(hindi_parquet_files))
for i, file in enumerate(hindi_parquet_files):
    print(i, file)

Hindi Parquet files: 5682
0 audio/Hindi/train-00000-of-05620.parquet
1 audio/Hindi/train-00001-of-05620.parquet
2 audio/Hindi/train-00002-of-05620.parquet
3 audio/Hindi/train-00003-of-05620.parquet
4 audio/Hindi/train-00004-of-05620.parquet
5 audio/Hindi/train-00005-of-05620.parquet
6 audio/Hindi/train-00006-of-05620.parquet
7 audio/Hindi/train-00007-of-05620.parquet
8 audio/Hindi/train-00008-of-05620.parquet
9 audio/Hindi/train-00009-of-05620.parquet
10 audio/Hindi/train-00010-of-05620.parquet
11 audio/Hindi/train-00011-of-05620.parquet
12 audio/Hindi/train-00012-of-05620.parquet
13 audio/Hindi/train-00013-of-05620.parquet
14 audio/Hindi/train-00014-of-05620.parquet
15 audio/Hindi/train-00015-of-05620.parquet
16 audio/Hindi/train-00016-of-05620.parquet
17 audio/Hindi/train-00017-of-05620.parquet
18 audio/Hindi/train-00018-of-05620.parquet
19 audio/Hindi/train-00019-of-05620.parquet
20 audio/Hindi/train-00020-of-05620.parquet
21 audio/Hindi/train-00021-of-05620.parquet
22 audio/Hindi/t

In [10]:
from huggingface_hub import hf_hub_download
hindi_file = hindi_parquet_files[0]
print("Selected file:")
print(hindi_file)

hindi_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=hindi_file,
    token=HF_TOKEN
)

print("\nDownloaded to:")
print(hindi_path)

Selected file:
audio/Hindi/train-00000-of-05620.parquet


audio/Hindi/train-00000-of-05620.parquet: reconstructing file:   0%|          |  0.00B /  495MB            

audio/Hindi/train-00000-of-05620.parquet: downloading bytes:           |  0.00B            


Downloaded to:
/root/.cache/huggingface/hub/datasets--ARTPARK-IISc--Vaani/snapshots/fe12c49bc61c083d5bb2092513fb6ec2e7ae72ee/audio/Hindi/train-00000-of-05620.parquet


In [59]:
dataset = Dataset.from_parquet(hindi_path)
print(dataset)

Dataset({
    features: ['audio', 'language', 'duration', 'speakerID', 'languagesKnown', 'gender', 'state', 'district', 'pincode', 'stay(years)', 'isTranscriptionAvailable', 'transcript', 'referenceImage', 'speakerImageHash', 'UtteranceSequenceID'],
    num_rows: 1799
})


In [60]:
df = dataset.to_pandas()

print("Total rows:", len(df))

print("\nColumns:")
print(df.columns.tolist())

Total rows: 1799

Columns:
['audio', 'language', 'duration', 'speakerID', 'languagesKnown', 'gender', 'state', 'district', 'pincode', 'stay(years)', 'isTranscriptionAvailable', 'transcript', 'referenceImage', 'speakerImageHash', 'UtteranceSequenceID']


In [54]:
print("\nTranscription availability values:")

print(
    df["isTranscriptionAvailable"].value_counts(dropna=False)
)


Transcription availability values:
isTranscriptionAvailable
No     1699
Yes     100
Name: count, dtype: int64


In [55]:
print("Transcript null values:")

print(df["transcript"].isna().sum())

print("\nFirst 10 transcripts:")

for i in range(min(10, len(df))):

    print("\nIndex:", i)
    print("Available:", df.iloc[i]["isTranscriptionAvailable"])
    print("Transcript:", repr(df.iloc[i]["transcript"]))

Transcript null values:
0

First 10 transcripts:

Index: 0
Available: No
Transcript: ''

Index: 1
Available: No
Transcript: ''

Index: 2
Available: No
Transcript: ''

Index: 3
Available: No
Transcript: ''

Index: 4
Available: No
Transcript: ''

Index: 5
Available: No
Transcript: ''

Index: 6
Available: No
Transcript: ''

Index: 7
Available: No
Transcript: ''

Index: 8
Available: No
Transcript: ''

Index: 9
Available: No
Transcript: ''


In [56]:
df_clean = df.copy()

# Keep rows where transcript exists
df_clean = df_clean[
    df_clean["transcript"].notna()
]

# Convert transcript to string
df_clean["transcript"] = df_clean["transcript"].astype(str)

# Remove empty transcripts
df_clean = df_clean[
    df_clean["transcript"].str.strip() != ""
]

# Remove string representations of missing values
df_clean = df_clean[
    ~df_clean["transcript"].str.lower().isin([
        "nan",
        "none",
        "null"
    ])
]

print("Original samples:", len(df))
print("Valid transcript samples:", len(df_clean))

Original samples: 1799
Valid transcript samples: 100


In [57]:
print(df_clean.shape)

print("\nFirst 5 valid samples:")

for i in range(min(5, len(df_clean))):

    print("\nSample:", i)
    print("Language:", df_clean.iloc[i]["language"])
    print("Transcript:", df_clean.iloc[i]["transcript"])

(100, 15)

First 5 valid samples:

Sample: 0
Language: Hindi
Transcript: <insect_noise> इस फोटो {photo} मे बहुत सारा पानी भरा हुआ नज़र आ रहा है एक व्यक्ति का सर दिख रहा है उसके साइड {side} मे बहुत सारी नाव खड़ी हुई नज़र आ रही है। उसेक -- </insect_noise>

Sample: 1
Language: Hindi
Transcript: <noise> इस चित्र में एक बड़े से जंगल की तस्वीर है जहां पे एक छोटा सा गुहा है सीढ़िया हैं। [breathing] </noise>

Sample: 2
Language: Hindi
Transcript: <noise> दीवारों का रंग जो है वो पीला और काला है [breathing] बहुत सारे लाइटो और लैम्पो {lamp} से सजाया गया है नीचे जमीन रंग जो है काला है। </noise>

Sample: 3
Language: Hindi
Transcript: यहा पर एक बिल्डिंग {building} दिखाई दे रही है बिल्डिंग {building} के सामने कुछ पेड़ लगे हुए हैं जो दिखाई दे रहे हैं।

Sample: 4
Language: Hindi
Transcript: यहा पर एक गाड़ी खड़ी हुई है जो दिखाई दे रही है एक गेट {gate} लगा है जो दिख  रहा है।


In [58]:
print("Total samples:", len(df))

print("\nNon-null transcripts:")
print(df["transcript"].notna().sum())

print("\nNon-empty transcripts:")

non_empty = df[
    df["transcript"].notna()
]

print(
    (
        non_empty["transcript"]
        .astype(str)
        .str.strip() != ""
    ).sum()
)

Total samples: 1799

Non-null transcripts:
1799

Non-empty transcripts:
100


In [61]:
MAX_SAMPLES = 500

df_subset = df_clean.sample(
    n=min(MAX_SAMPLES, len(df_clean)),
    random_state=42
)

df_subset = df_subset.reset_index(drop=True)

print("Subset size:", len(df_subset))

Subset size: 100


In [62]:
print(df_subset.columns.tolist())

['audio', 'language', 'duration', 'speakerID', 'languagesKnown', 'gender', 'state', 'district', 'pincode', 'stay(years)', 'isTranscriptionAvailable', 'transcript', 'referenceImage', 'speakerImageHash', 'UtteranceSequenceID']


In [63]:
df_subset = df_subset[
    ["audio", "transcript"]
]

print(df_subset.head())

                                               audio  \
0  {'bytes': b'RIFF\x04#\x07\x00WAVEfmt \x10\x00\...   
1  {'bytes': b'RIFFdm\x03\x00WAVEfmt \x10\x00\x00...   
2  {'bytes': b'RIFFd=\x03\x00WAVEfmt \x10\x00\x00...   
3  {'bytes': b'RIFFd\xec\x04\x00WAVEfmt \x10\x00\...   
4  {'bytes': b'RIFF\xa4*\x05\x00WAVEfmt \x10\x00\...   

                                          transcript  
0  <walking>यह काफी लम्बा रोड {road} दिखाई दे रहा...  
1  ऊपर काफी सारे पंखे हमे लटके दिखाई दे रहे है नी...  
2  <noise> [unintelligible] अलमारियों मे शीतल के ...  
3  <noise> एक जगह का चित्र है [breathing] जिसमें ...  
4  दिखाई दे रही एक और [unintelligible] सफ़ेद दिखाई...  


In [64]:
from datasets import Dataset

dataset_subset = Dataset.from_pandas(
    df_subset,
    preserve_index=False
)

print(dataset_subset)
print(dataset_subset.column_names)

Dataset({
    features: ['audio', 'transcript'],
    num_rows: 100
})
['audio', 'transcript']


In [65]:
print("Dataset size:", len(dataset_subset))

print("\nFirst sample keys:")
print(dataset_subset[0].keys())

print("\nTranscript:")
print(dataset_subset[0]["transcript"])

Dataset size: 100

First sample keys:
dict_keys(['audio', 'transcript'])

Transcript:
<walking>यह काफी लम्बा रोड {road} दिखाई दे रहा है ऊपर जिसे हम ब्रिज {bridge} भी कह सकते है [sniffing] नीचे पानी ही पानी है [breathing] और ब्रिज {bridge} के ऊपर एक और कुछ [breathing] बनाया गया है जो कि सीमेंट {cement} से बनाया है। [nose_blowing] </walking>


In [66]:
dataset_split = dataset_subset.train_test_split(
    test_size=0.2,
    seed=42
)

print("Train samples:", len(dataset_split["train"]))
print("Test samples:", len(dataset_split["test"]))

Train samples: 80
Test samples: 20


In [67]:
train_dataset = dataset_split["train"]

eval_dataset = dataset_split["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['audio', 'transcript'],
    num_rows: 80
})
Dataset({
    features: ['audio', 'transcript'],
    num_rows: 20
})


In [68]:
print("Train rows:", len(train_dataset))
print("Eval rows:", len(eval_dataset))

print("\nTrain columns:")
print(train_dataset.column_names)

Train rows: 80
Eval rows: 20

Train columns:
['audio', 'transcript']


In [75]:
print(
    df_clean["language"].value_counts()
)

language
Hindi    100
Name: count, dtype: int64


In [77]:
def prepare_dataset(batch):

    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        batch["transcript"]
    ).input_ids

    return batch

In [78]:
train_dataset_processed = train_dataset.map(
    prepare_dataset,
    remove_columns=train_dataset.column_names
)

eval_dataset_processed = eval_dataset.map(
    prepare_dataset,
    remove_columns=eval_dataset.column_names
)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

KeyError: 'array'